# PID-UNet v7 — train từ đầu, lưu trên Drive
Chọn **Runtime → Change runtime type → GPU** rồi chạy các ô theo thứ tự. Model có 758.257 tham số, kế thừa encoder v2 và giữ nguyên loss. Chưa có kết quả thực nghiệm chứng minh vượt EDANet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
assert Path('/content/drive/MyDrive').is_dir(), 'Drive chưa được mount'
OUTPUT_ROOT = Path('/content/drive/MyDrive/polyp_runs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO = Path('/content/polyp-segmentation')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/nguyenphucvinh111/polyp-segmentation.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trước khi train'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

Lần đầu để `RESUME = False`. Sau khi ngắt kết nối, chạy lại các ô trên rồi đổi thành `True` để tiếp tục từ epoch đã lưu. Giữ cùng seed, RUN_NAME và số epoch tổng. Muốn train mới thì đặt RUN_NAME khác; chương trình sẽ chặn ghi đè kết quả cũ.

In [ ]:
RUN_NAME = 'pid_unet_v7'
RESUME = False
SEED = 42
command = [sys.executable, 'train.py', '--model', 'pid_unet_v7',
           '--loss', 'structure_loss', '--epochs', '100', '--batchsize', '8',
           '--seed', str(SEED), '--run_name', RUN_NAME, '--output_root', str(OUTPUT_ROOT)]
if RESUME:
    checkpoint = OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'last.pth'
    assert checkpoint.is_file(), f'Không thấy checkpoint: {checkpoint}'
    command.extend(['--resume', str(checkpoint)])
subprocess.run(command, check=True)

In [ ]:
metrics = OUTPUT_ROOT / 'results' / RUN_NAME / 'metrics.csv'
assert metrics.is_file(), 'Cần hoàn thành benchmark trước'
subprocess.run([sys.executable, 'compare_edanet.py', '--metrics', str(metrics)], check=True)
print('Best checkpoint:', OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'best.pth')
print('Resume checkpoint:', OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'last.pth')
print('Results:', metrics)